# BERT sentiment notebook fixed for Google Colab (Python 3.12)

This version was adjusted to work reliably with the current Colab runtime and the Keras 3 / TensorFlow Hub compatibility changes.

## What changed
- enables **legacy Keras 2 mode** before importing TensorFlow
- pins a **Python 3.12 compatible TensorFlow stack**
- removes the broken `keras.layers.preprocessing` pattern
- removes the dependency on `tf-models-official` for the optimizer
- makes dataset extraction and model saving more robust
- wraps `plot_model()` so the notebook does not fail if Graphviz is missing

## How to run
1. Run the **first code cell**.  
   If package installation is needed, the runtime will restart automatically once.
2. After reconnecting, continue running the notebook from the next cell, or simply use **Run all** again.

In [ ]:
import os
import sys
import signal
import subprocess
import importlib.metadata as md

# Must be set BEFORE importing tensorflow
os.environ["TF_USE_LEGACY_KERAS"] = "1"

REQUIRED = {
    "tensorflow": "2.19.0",
    "tf-keras": "2.19.0",
    "tensorflow-text": "2.19.0",
    "tensorflow-hub": "0.16.1",
}

def installed_version(package_name):
    try:
        return md.version(package_name)
    except md.PackageNotFoundError:
        return None

current_versions = {pkg: installed_version(pkg) for pkg in REQUIRED}
print("Current package versions:")
for pkg, ver in current_versions.items():
    print(f"  {pkg}: {ver}")

to_install = [f"{pkg}=={ver}" for pkg, ver in REQUIRED.items() if current_versions[pkg] != ver]

if to_install:
    print("\nInstalling / aligning packages:")
    for pkg in to_install:
        print(" ", pkg)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *to_install])
    print("\nRuntime will restart once to activate the new stack...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("\nEnvironment already prepared. Continue with the next cell.")

Current package versions:
  tensorflow: 2.19.0
  tf-keras: 2.19.0
  tensorflow-text: 2.19.0
  tensorflow-hub: 0.16.1

Environment already prepared. Continue with the next cell.


In [ ]:
import os
import shutil
import random

import numpy as np
import tensorflow as tf
import tensorflow_hub as hub
import tensorflow_text as text
import matplotlib.pyplot as plt

tf.get_logger().setLevel('ERROR')
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

print("TF:", tf.__version__)
print("NumPy:", np.__version__)
print("Legacy Keras:", os.environ.get("TF_USE_LEGACY_KERAS"))

TF: 2.19.0
NumPy: 2.0.2
Legacy Keras: 1


In [ ]:
url = 'https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz'
dataset = tf.keras.utils.get_file(
    fname='aclImdb_v1.tar.gz',
    origin=url,
    untar=True,
    cache_dir='.',
    cache_subdir=''
)

# TensorFlow usually extracts to ./datasets/aclImdb, but we also keep a fallback
default_dataset_dir = os.path.join(os.path.dirname(dataset), 'aclImdb')
fallback_dataset_dir = '/content/aclImdb_v1_extracted/aclImdb'
dataset_dir = default_dataset_dir if os.path.isdir(default_dataset_dir) else fallback_dataset_dir

train_dir = os.path.join(dataset_dir, 'train')
test_dir = os.path.join(dataset_dir, 'test')
remove_dir = os.path.join(train_dir, 'unsup')

if os.path.isdir(remove_dir):
    shutil.rmtree(remove_dir)

print("Dataset directory:", dataset_dir)
print("Train directory  :", train_dir)
print("Test directory   :", test_dir)

84125825/84125825 [==============================] - 18s 0us/step
Dataset directory: ./aclImdb
Train directory  : ./aclImdb/train
Test directory   : ./aclImdb/test


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = 32
seed = 42

raw_train_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='training',
    seed=seed
)
class_names = raw_train_ds.class_names
train_ds = raw_train_ds.cache().prefetch(buffer_size=AUTOTUNE)

print("Class names:", class_names)

Found 25000 files belonging to 2 classes.
Using 20000 files for training.
Class names: ['neg', 'pos']


In [ ]:
val_ds = tf.keras.utils.text_dataset_from_directory(
    train_dir,
    batch_size=batch_size,
    validation_split=0.2,
    subset='validation',
    seed=seed
)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

test_ds = tf.keras.utils.text_dataset_from_directory(
    test_dir,
    batch_size=batch_size
)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

Found 25000 files belonging to 2 classes.
Using 5000 files for validation.
Found 25000 files belonging to 2 classes.


In [ ]:
for text_batch, label_batch in train_ds.take(1):
  for i in range(3):
    print(f'Recenzja: {text_batch.numpy()[i]}')
    label = label_batch.numpy()[i]
    print(f'Etykieta: {label}({class_names[label]})')

Recenzja: b"Mild Spoilers<br /><br />In the near future, Arnold stars as Ben Richards, a wrongly convicted man coerced into playing 'The Running Man', a deadly TV game show where people have to keep moving to try and escape brutal deaths at the hands of the 'Stalkers'. Of course, people are expected to die eventually and its up to Arnold to prove the system wrong.<br /><br />I haven't read the Stephen King book, but this is a great film regardless, one of Arnold's best. He does what he does best in the action man role, delivering death with unforgettable one-liners. Classics are probably the 'He was a real pain in the neck' after strangling a guy with barb wire, and 'He had to split!', referring to whereabouts he just chain sawed someone vertically. Dawson is perfectly irritating as the TV presenter, and all the 'Stalkers' are suitably camp. The action is violent, but its an action film. That's the point. The film is fast paced, and at 90 minutes it doesn't overstay its welcome. <br />

In [ ]:
bert_model_name = 'small_bert/bert_en_uncased_L-4_H-512_A-8'

map_name_to_handle = {
    'bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_L-12_H-768_A-12/3',
    'bert_en_cased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_cased_L-12_H-768_A-12/3',
    'bert_multi_cased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_multi_cased_L-12_H-768_A-12/3',
    'small_bert/bert_en_uncased_L-2_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-2_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-2_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-2_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-2_H-768_A-12/1',
    'small_bert/bert_en_uncased_L-4_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-4_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-4_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-4_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-768_A-12/1',
    'small_bert/bert_en_uncased_L-6_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-6_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-6_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-6_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-6_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-6_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-6_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-6_H-768_A-12/1',
    'small_bert/bert_en_uncased_L-8_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-8_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-8_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-8_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-8_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-8_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-8_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-8_H-768_A-12/1',
    'small_bert/bert_en_uncased_L-10_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-10_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-10_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-10_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-10_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-10_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-10_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-10_H-768_A-12/1',
    'small_bert/bert_en_uncased_L-12_H-128_A-2':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-12_H-128_A-2/1',
    'small_bert/bert_en_uncased_L-12_H-256_A-4':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-12_H-256_A-4/1',
    'small_bert/bert_en_uncased_L-12_H-512_A-8':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-12_H-512_A-8/1',
    'small_bert/bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-12_H-768_A-12/1',
    'albert_en_base':
        'https://tfhub.dev/tensorflow/albert_en_base/2',
    'electra_small':
        'https://tfhub.dev/google/electra_small/2',
    'electra_base':
        'https://tfhub.dev/google/electra_base/2',
    'experts_pubmed':
        'https://tfhub.dev/google/experts/bert/pubmed/2',
    'experts_wiki_books':
        'https://tfhub.dev/google/experts/bert/wiki_books/2',
    'talking-heads_base':
        'https://tfhub.dev/tensorflow/talkheads_ggelu_bert_en_base/1',
}

map_model_to_preprocess = {
    'bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'bert_en_cased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_cased_preprocess/3',
    'small_bert/bert_en_uncased_L-2_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-2_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-2_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-2_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-4_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-4_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-4_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-4_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-6_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-6_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-6_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-6_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-8_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-8_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-8_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-8_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-10_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-10_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-10_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-10_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-12_H-128_A-2':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-12_H-256_A-4':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-12_H-512_A-8':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'small_bert/bert_en_uncased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'bert_multi_cased_L-12_H-768_A-12':
        'https://tfhub.dev/tensorflow/bert_multi_cased_preprocess/3',
    'albert_en_base':
        'https://tfhub.dev/tensorflow/albert_en_preprocess/3',
    'electra_small':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'electra_base':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'experts_pubmed':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'experts_wiki_books':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
    'talking-heads_base':
        'https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3',
}

tfhub_handle_encoder = map_name_to_handle[bert_model_name]
tfhub_handle_preprocess = map_model_to_preprocess[bert_model_name]

print(f'BERT model selected           : {tfhub_handle_encoder}')
print(f'Preprocess model auto-selected: {tfhub_handle_preprocess}')


BERT model selected           : https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1
Preprocess model auto-selected: https://tfhub.dev/tensorflow/bert_en_uncased_preprocess/3


In [ ]:
bert_preprocess_model = hub.KerasLayer(tfhub_handle_preprocess)

In [ ]:
text_test = ['this is such an amazing movie!']
text_preprocessed = bert_preprocess_model(text_test)

print(f'Keys      : {list(text_preprocessed.keys())}')
print(f'Shape     : {text_preprocessed["input_word_ids"].shape}')
print(f'Word Ids  : {text_preprocessed["input_word_ids"][0,:12]}')
print(f'Input Mask: {text_preprocessed["input_mask"][0,:12]}')
print(f'Type Ids  : {text_preprocessed["input_type_ids"][0,:12]}')

Keys      : ['input_type_ids', 'input_mask', 'input_word_ids']
Shape     : (1, 128)
Word Ids  : [ 101 2023 2003 2107 2019 6429 3185  999  102    0    0    0]
Input Mask: [1 1 1 1 1 1 1 1 1 0 0 0]
Type Ids  : [0 0 0 0 0 0 0 0 0 0 0 0]


In [ ]:
bert_model = hub.KerasLayer(tfhub_handle_encoder)

In [ ]:
best_results = bert_model(text_preprocessed)
print(f'Loaded BERT: {tfhub_handle_encoder}')
print(f'Pooled Outputs Shape: {best_results["pooled_output"].shape}')
print(f'Pooled Outputs Values: {best_results["pooled_output"][0,:12]}')
print(f'Sequenced Outputs Shape: {best_results["sequence_output"].shape}')
print(f'Sequenced Outputs Values: {best_results["sequence_output"][0,:12]}')

Loaded BERT: https://tfhub.dev/tensorflow/small_bert/bert_en_uncased_L-4_H-512_A-8/1
Pooled Outputs Shape: (1, 512)
Pooled Outputs Values: [ 0.76262915  0.9928099  -0.18611892  0.3667383   0.15233688  0.6550449
  0.9681154  -0.948627    0.00216142 -0.9877732   0.06842722 -0.97630596]
Sequenced Outputs Shape: (1, 128, 512)
Sequenced Outputs Values: [[-0.28946328  0.34321254  0.33231536 ...  0.21300852  0.7102075
  -0.05771133]
 [-0.28742024  0.31981018 -0.23018624 ...  0.58455026 -0.2132977
   0.7269201 ]
 [-0.66157013  0.68876845 -0.87432915 ...  0.10877264 -0.26173204
   0.4785539 ]
 ...
 [-0.22561169 -0.2892565  -0.0706441  ...  0.47566032  0.8327713
   0.40025395]
 [-0.29824263 -0.2747311  -0.0545048  ...  0.48849747  1.0955359
   0.18163365]
 [-0.443782    0.00930733  0.0722375  ...  0.17290099  1.1833246
   0.07898037]]


In [ ]:
def build_classifier_model():
    text_input = tf.keras.layers.Input(shape=(), dtype=tf.string, name='text')

    preprocessing_layer = hub.KerasLayer(
        tfhub_handle_preprocess,
        name='preprocessing'
    )
    encoder_inputs = preprocessing_layer(text_input)

    encoder = hub.KerasLayer(
        tfhub_handle_encoder,
        trainable=True,
        name='BERT_encoder'
    )
    outputs = encoder(encoder_inputs)

    net = outputs['pooled_output']
    net = tf.keras.layers.Dropout(0.1)(net)
    net = tf.keras.layers.Dense(1, activation=None, name='classifier')(net)

    return tf.keras.Model(text_input, net)

In [ ]:
text_test = ["this is such an amazing movie"]

classifier_model = build_classifier_model()
bert_raw_result = classifier_model(tf.constant(text_test))
print(tf.sigmoid(bert_raw_result))

tf.Tensor([[0.80802417]], shape=(1, 1), dtype=float32)


In [ ]:
try:
    tf.keras.utils.plot_model(classifier_model, show_shapes=True, dpi=72)
except Exception as e:
    print("plot_model() skipped:", e)

In [ ]:
loss = tf.keras.losses.BinaryCrossentropy(from_logits=True)
metrics = tf.metrics.BinaryAccuracy()

In [ ]:
epochs = 5
steps_per_epoch = int(tf.data.experimental.cardinality(train_ds).numpy())
num_train_steps = steps_per_epoch * epochs

init_lr = 3e-5
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=init_lr,
    weight_decay=0.01
)

print("Steps per epoch :", steps_per_epoch)
print("Train steps     :", num_train_steps)
print("Learning rate   :", init_lr)

Steps per epoch : 625
Train steps     : 3125
Learning rate   : 3e-05


In [ ]:
classifier_model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics,
    jit_compile=False
)

In [ ]:
print('trening modelu z premodelem BERT')
history = classifier_model.fit(x = train_ds,
                               validation_data = val_ds,
                               epochs=epochs)

trening modelu z premodelem BERT
Epoch 1/5
625/625 [==============================] - 186s 267ms/step - loss: 0.4401 - binary_accuracy: 0.7825 - val_loss: 0.3548 - val_binary_accuracy: 0.8444
Epoch 2/5
625/625 [==============================] - 150s 240ms/step - loss: 0.3083 - binary_accuracy: 0.8631 - val_loss: 0.3664 - val_binary_accuracy: 0.8484
Epoch 3/5
625/625 [==============================] - 152s 244ms/step - loss: 0.2198 - binary_accuracy: 0.9091 - val_loss: 0.4001 - val_binary_accuracy: 0.8528
Epoch 4/5
625/625 [==============================] - 152s 244ms/step - loss: 0.1476 - binary_accuracy: 0.9432 - val_loss: 0.4913 - val_binary_accuracy: 0.8440
Epoch 5/5
625/625 [==============================] - 151s 242ms/step - loss: 0.0961 - binary_accuracy: 0.9645 - val_loss: 0.6584 - val_binary_accuracy: 0.8370


In [ ]:
loss,accuracy = classifier_model.evaluate(test_ds)
print(f'Loss: {loss}')
print(f'Accuracy: {accuracy}')

782/782 [==============================] - 72s 92ms/step - loss: 0.6437 - binary_accuracy: 0.8400
Loss: 0.6436683535575867
Accuracy: 0.839959979057312


In [ ]:
dataset_name = 'imdb'
save_model_path = f'./{dataset_name}_bert_savedmodel'

if os.path.isdir(save_model_path):
    shutil.rmtree(save_model_path)

tf.saved_model.save(classifier_model, save_model_path)
print("Model saved to:", save_model_path)

Model saved to: ./imdb_bert_savedmodel


In [ ]:
# ładowanie ponowne modelu
reloaded_model = tf.saved_model.load(save_model_path)
print("Reloaded model ready.")

Reloaded model ready.


In [ ]:
def print_my_examples(inputs,results):
  result_for_printing = [f'input: {inputs[i]:<30}: score: {results[i][0]:.6f}' for i in range(len(inputs))]
  print(*result_for_printing,sep='\n')
  print()

examples = [
    'this is sach an amazing movie!',
    'The movie was great!',
    'The movie was meh!',
    'The movie was okish.',
    'The movie was teribble...',
    'The movie was ok.'
]

reloaded_results = tf.sigmoid(reloaded_model(tf.constant(examples)))
original_results = tf.sigmoid(classifier_model(tf.constant(examples)))

print('Wyniki z zapisanego modelu:')
print_my_examples(examples,reloaded_results)
print('Wyniki z modelu w pamięci:')
print_my_examples(examples,original_results)

Wyniki z zapisanego modelu:
input: this is sach an amazing movie!: score: 0.999524
input: The movie was great!          : score: 0.994138
input: The movie was meh!            : score: 0.993064
input: The movie was okish.          : score: 0.012435
input: The movie was teribble...     : score: 0.000816
input: The movie was ok.             : score: 0.125235

Wyniki z modelu w pamięci:
input: this is sach an amazing movie!: score: 0.999524
input: The movie was great!          : score: 0.994138
input: The movie was meh!            : score: 0.993064
input: The movie was okish.          : score: 0.012435
input: The movie was teribble...     : score: 0.000816
input: The movie was ok.             : score: 0.125235

